In [171]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [172]:
datasets = "..\\..\\datasets\\"
models = "..\\..\\models\\"

IMAGES_PATH = f"{datasets}/coco/train2014"  # Directory with training images
VAL_IMAGES_PATH = f"{datasets}/coco/val2014"  # Directory with validation images
CAPTIONS_PATH = f"{datasets}/coco/annotations/annotations/captions_train2014.json"  # Caption file
VAL_CAPTIONS_PATH = f"{datasets}/coco/annotations/annotations/captions_val2014.json"  # Validation caption file
TEST_IMAGES_PATH = "..\\test_images"  # Directory with test images

In [173]:
import tqdm
import nltk 
from collections import Counter
from vocabulary_class import Vocabulary
nltk.download('punkt')
import json

tokens = []
counter = Counter()

import csv
import string

tokens = []
counter = Counter()

def build_vocab(json_path, threshold=5, limit=None):
    with open(json_path, 'r') as f:
        data = json.load(f) 

    counter = Counter()
    count =0
    image_captions ={}

    for ann in tqdm.tqdm(data['annotations']):
        # print(ann)
        img_name = ann['image_id']
        caption = ann['caption'].lower()
        
        if img_name not in image_captions:
                image_captions[img_name] = []
        image_captions[img_name].append(caption)

        
        tokens = nltk.tokenize.word_tokenize(caption)
        counter.update(tokens)
        count +=1
        if limit and count >= limit:
            break
    
    vocab = Vocabulary()
    for word, cnt in counter.items():
        if cnt >= threshold:
            vocab.add_word(word)
    
    return vocab

[nltk_data] Downloading package punkt to /home/madhu-
[nltk_data]     thiramdas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [174]:
vocab = build_vocab(CAPTIONS_PATH, threshold=5)
print("Total vocabulary size:", len(vocab))

100%|██████████| 202654/202654 [00:08<00:00, 24500.60it/s]

Total vocabulary size: 6507


In [175]:
import torch
from torch.utils.data import Dataset
from pycocotools.coco import COCO
from PIL import Image
import nltk
import os

class CocoDatasetClass(Dataset):
    def __init__(self, root, json_path, vocab, transform=None, max_caps=None):
        self.root = root
        self.coco = COCO(json_path)
        self.vocab = vocab
        self.transform = transform

        self.samples = []  # (img_id, caption)
        img_ids = self.coco.getImgIds()

        for img_id in img_ids:
            ann_ids = self.coco.getAnnIds(imgIds=img_id)
            anns = self.coco.loadAnns(ann_ids)
            for ann in anns:
                self.samples.append((img_id, ann["caption"]))

        if max_caps:
            self.samples = self.samples[:max_caps]

        # cache file names
        self.id_to_file = {img_id: self.coco.loadImgs(img_id)[0]["file_name"] for img_id in img_ids}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_id, caption = self.samples[index]
        img_path = os.path.join(self.root, self.id_to_file[img_id])

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        tokens = nltk.word_tokenize(caption.lower())
        cap_ids = [self.vocab.word2idx["<start>"]]
        cap_ids += [self.vocab.word2idx.get(t, self.vocab.word2idx["<unk>"]) for t in tokens]
        cap_ids.append(self.vocab.word2idx["<end>"])

        return image, torch.tensor(cap_ids, dtype=torch.long), int(img_id)


In [192]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms

from torch.nn.utils.rnn import pad_sequence
import torch

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), 
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])


def collate_fn(batch):
    images, captions , ids= zip(*batch)

    images = torch.stack(images, 0)

    captions = pad_sequence(
        captions,
        batch_first=True,
        padding_value=0
    )

    return images, captions, ids

from torch.utils.data import DataLoader

test_dataset = CocoDatasetClass(
    root=IMAGES_PATH,
    json_path=CAPTIONS_PATH,
    vocab=vocab,
    transform=transform
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=12,
    collate_fn= collate_fn
)


print(len(test_dataset))
image, caption , ids= test_dataset[0]

print(type(image))
print(image.shape)        # after transform
print(caption)
print(len(caption))
print(ids)

loading annotations into memory...
Done (t=0.13s)
creating index...
index created!
202654
<class 'torch.Tensor'>
torch.Size([3, 224, 224])
tensor([   1,    4,  171,   22,    4,   36,  833,   40,    4,   60, 1437,   40,
           4, 2029,   41,   19,    2])
17
391895


In [177]:
from model import TransformerEncoderViT
from model import TransformerDecoder

vocab = torch.load("models/vocab.pkl", weights_only=False)
encoder = TransformerEncoderViT().to(device)
decoder = TransformerDecoder(embed_size=256, vocab_size=len(vocab)).to(device)

encoder.load_state_dict(torch.load("models/encoder.pth", map_location=device))
decoder.load_state_dict(torch.load("models/decoder.pth", map_location=device))

<All keys matched successfully>

In [178]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

@torch.no_grad()
def beam_decode_from_memory(memory, decoder, vocab, max_len=30, beam_size=5, length_penalty=0.7):
    bos = vocab.word2idx["<start>"]
    eos = vocab.word2idx["<end>"]

    # beams: list of (seq_tensor [1,t], score_float, ended_bool)
    seq0 = torch.tensor([[bos]], device=memory.device, dtype=torch.long)
    beams = [(seq0, 0.0, False)]

    for _ in range(max_len):
        all_candidates = []

        # Early exit if all beams ended
        if all(ended for _, _, ended in beams):
            break

        for seq, score, ended in beams:
            if ended:
                all_candidates.append((seq, score, True))
                continue

            with autocast():  # decoder can benefit too
                logits = decoder(memory, seq)          # [1, t, V] (assumed)
                log_probs = F.log_softmax(logits[:, -1, :], dim=-1)  # [1, V]

            topk_logp, topk_idx = torch.topk(log_probs, beam_size, dim=-1)  # [1,k], [1,k]

            for k in range(beam_size):
                tok = topk_idx[0, k].view(1, 1)  # [1,1]
                next_seq = torch.cat([seq, tok], dim=1)

                new_score = score + float(topk_logp[0, k])
                new_ended = (int(tok.item()) == eos)

                all_candidates.append((next_seq, new_score, new_ended))

        # length-penalized sorting
        def rank(item):
            seq, sc, _ = item
            L = seq.size(1)
            return sc / (L ** length_penalty)

        beams = sorted(all_candidates, key=rank, reverse=True)[:beam_size]

    best_seq = beams[0][0].squeeze(0).tolist()

    words = []
    for idx in best_seq:
        w = vocab.idx2word[idx]
        if w in ("<start>", "<pad>"):
            continue
        if w == "<end>":
            break
        words.append(w)

    return " ".join(words)

In [179]:
from torch.cuda.amp import autocast

encoder.eval()
decoder.eval()
torch.backends.cudnn.benchmark = True

references, hypotheses = [], []

with torch.inference_mode():
    for images, captions, ids in tqdm.tqdm(test_loader):
        images = images.to(device, non_blocking=True)

        # Encode the whole batch once
        with autocast():
            batch_memory = encoder(images)   # [B, N, D] or similar

        for i in range(images.size(0)):
            img_id = int(ids[i])

            # refs
            ann_ids = test_dataset.coco.getAnnIds(imgIds=img_id)
            anns = test_dataset.coco.loadAnns(ann_ids)
            references.append([a["caption"].lower().split() for a in anns])

            # hyp (decode using precomputed memory)
            hyp = beam_decode_from_memory(
                batch_memory[i:i+1], decoder, vocab,
                max_len=30, beam_size=5, length_penalty=0.7
            )
            hypotheses.append(hyp.split())

  0%|          | 0/4 [00:00<?, ?it/s]/tmp/ipykernel_19590/2497506824.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_19590/571096737.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # decoder can benefit too
/home/madhu-thiramdas/ai-work/ai-venv-py310/lib/python3.10/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|██████████| 4/4 [00:19<00:00,  4.82s/it]


In [180]:
import numpy as np

print("HYP:", hypotheses[0])
print("REF:", references[0][0])
print("Avg hyp length:", np.mean([len(h) for h in hypotheses]))

HYP: ['a', 'man', 'riding', 'a', 'bike', 'down', 'the', 'side', 'of', 'a', 'dirt', 'road', '.']
REF: ['a', 'man', 'with', 'a', 'red', 'helmet', 'on', 'a', 'small', 'moped', 'on', 'a', 'dirt', 'road.']
Avg hyp length: 9.75


In [181]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
smooth = SmoothingFunction().method4

bleu4 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smooth
)

print(f"Smoothed BLEU-4: {bleu4:.4f}")

Smoothed BLEU-4: 0.0992


In [182]:
bleu1 = corpus_bleu(references, hypotheses, weights=(1,0,0,0))
bleu2 = corpus_bleu(references, hypotheses, weights=(0.5,0.5,0,0))
print("BLEU-1:", bleu1)
print("BLEU-2:", bleu2)

BLEU-1: 0.5794871794871795
BLEU-2: 0.39450437213508754


In [183]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_scores = []

for hyp, refs in zip(hypotheses, references):
    hyp_str = " ".join(hyp)
    best = max(
        scorer.score(" ".join(r), hyp_str)["rougeL"].fmeasure
        for r in refs
    )
    rouge_scores.append(best)

print(f"ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.4f}")

ROUGE-L: 0.4558


In [184]:
from jiwer import wer
def best_wer(hyp, refs):
    hyp_str = " ".join(hyp)
    return min(wer(" ".join(r), hyp_str) for r in refs)

wer_score = sum(
    best_wer(h, r) for h, r in zip(hypotheses, references)
) / len(hypotheses)

print(f"WER: {wer_score:.4f}")

WER: 0.7084


In [193]:
import torch
import torch.nn.functional as F
import math
import torch.nn as nn

def recall_at_k(similarity, k):
    """
    similarity: [N, N] similarity matrix
    """
    topk = similarity.topk(k, dim=1).indices
    targets = torch.arange(similarity.size(0)).unsqueeze(1).to(similarity.device)
    correct = (topk == targets).any(dim=1)
    return correct.float().mean().item()

def extract_image_embeddings(dataloader, encoder, decoder,  device):
    encoder.eval()
    decoder.eval()

    all_embs = []
    all_ids = []
    with torch.no_grad():
        for images, _ , image_ids in tqdm.tqdm(dataloader):
            images = images.to(device)
            enc_out = encoder(images)              # [B, 196, 768]
            enc_out = decoder.enc_proj(enc_out)    # ✅ project to 256

            img_emb = enc_out.mean(dim=1)          # [B, 256]
            img_emb = F.normalize(img_emb, dim=1)

            all_embs.append(img_emb.cpu())
            all_ids.extend(image_ids)

        return torch.cat(all_embs, dim=0), all_ids
@torch.no_grad()
def extract_text_embeddings(encoder, decoder, dataloader, vocab, device):
    encoder.eval()
    decoder.eval()

    all_embeddings = []
    all_ids = []

    pad_idx = vocab.word2idx["<pad>"]

    for images, captions, image_ids in tqdm.tqdm(dataloader):
        images = images.to(device)
        captions = captions.to(device)

        # ---- Encode images ----
        enc = encoder(images)                 # [B, 196, 768]
        memory = decoder.enc_proj(enc)        # [B, 196, 256]  ✅ important

        B, T = captions.shape

        # ---- Build tgt embeddings (scale then pos) ----
        d_model = decoder.embed.embedding_dim
        x = decoder.embed(captions) * math.sqrt(d_model)
        x = decoder.pos(x)

        # ---- masks ----
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(T).to(device)
        padding_mask = (captions == pad_idx)

        # ---- decode hidden states ----
        out = decoder.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=padding_mask,
        )  # [B, T, 256]

        # ---- last non-pad token ----
        lengths = (~padding_mask).sum(dim=1) - 1
        lengths = torch.clamp(lengths, min=0)   # safety

        sent_emb = out[torch.arange(B, device=device), lengths]  # [B, 256]
        sent_emb = F.normalize(sent_emb, dim=1)

        all_embeddings.append(sent_emb.cpu())
        all_ids.extend(image_ids)

    return torch.cat(all_embeddings, dim=0), all_ids


In [194]:

def image_to_text_retrieval(image_emb, text_emb, image_ids, text_ids, batch_size=512):
    image_emb = F.normalize(image_emb, dim=1)
    text_emb  = F.normalize(text_emb, dim=1)
    
    image_emb = image_emb.cuda()
    text_emb  = text_emb.cuda()

     # build ID → index map for text embeddings
    text_id_to_index = {tid: idx for idx, tid in enumerate(text_ids)}

    N = image_emb.size(0)
    ranks = []

    for i in range(0, N, batch_size):
        img_batch = image_emb[i:i+batch_size]           # [B, D]
        sim = img_batch @ text_emb.T                     # [B, N]

        batch_image_ids = image_ids[i:i+batch_size]
        
        sorted_idx = sim.argsort(dim=1, descending=True)

        for j, img_id in enumerate(batch_image_ids):
            gt_index = text_id_to_index[img_id]

            rank = (sorted_idx[j] == gt_index).nonzero(as_tuple=True)[0].item()
            ranks.append(rank)

    ranks = torch.tensor(ranks, device=image_emb.device)
    return {
        "R@1":  (ranks < 1).float().mean().item(),
        "R@5":  (ranks < 5).float().mean().item(),
        "R@10": (ranks < 10).float().mean().item()
    }

def text_to_image_retrieval(image_emb, text_emb, image_ids, text_ids, batch_size=512):
    image_emb = F.normalize(image_emb, dim=1)
    text_emb  = F.normalize(text_emb, dim=1)

    text_emb  = text_emb.cuda()
    image_emb = image_emb.cuda()

    # build ID → index map for images
    image_id_to_index = {iid: idx for idx, iid in enumerate(image_ids)}

    N = text_emb.size(0)
    ranks = []

    for i in range(0, N, batch_size):
        txt_batch = text_emb[i:i+batch_size]     # [B, D]
        sim = txt_batch @ image_emb.T             # [B, N_image]

    
        batch_text_ids = text_ids[i:i+batch_size]
        sorted_idx = sim.argsort(dim=1, descending=True)

        for j, txt_id in enumerate(batch_text_ids):
            gt_index = image_id_to_index[txt_id]  # ✅ correct GT image

            rank = (sorted_idx[j] == gt_index).nonzero(as_tuple=True)[0].item()
            ranks.append(rank)

    ranks = torch.tensor(ranks, device=text_emb.device)
    return {
        "R@1":  (ranks < 1).float().mean().item(),
        "R@5":  (ranks < 5).float().mean().item(),
        "R@10": (ranks < 10).float().mean().item()
    }

def text_to_text_retrieval(text_emb, text_ids, batch_size=512):
    text_emb  = F.normalize(text_emb, dim=1)

    text_emb = text_emb.cuda()

    # build image_id → list of text indices
    id_to_indices = {}
    for idx, img_id in enumerate(text_ids):
        id_to_indices.setdefault(img_id, []).append(idx)

    N = text_emb.size(0)
    ranks = []

    for i in range(0, N, batch_size):
        txt_batch = text_emb[i:i+batch_size]     # [B, D]
        sim = txt_batch @ text_emb.T              # [B, N]

        sorted_idx = sim.argsort(dim=1, descending=True)

        for j in range(sorted_idx.size(0)):
            query_idx = i + j
            img_id = text_ids[query_idx]

            # valid GT indices = same image_id, excluding itself
            gt_indices = [idx for idx in id_to_indices[img_id] if idx != query_idx]

            if len(gt_indices) == 0:
                continue 
            # find best (lowest) rank among all GT indices
            rank = min(
                (sorted_idx[j] == gt).nonzero(as_tuple=True)[0].item()
                for gt in gt_indices
            )
            ranks.append(rank)

    ranks = torch.tensor(ranks, device=text_emb.device)

    return {
        "R@1":  (ranks < 1).float().mean().item(),
        "R@5":  (ranks < 5).float().mean().item(),
        "R@10": (ranks < 10).float().mean().item()
    }

def image_to_image_retrieval(image_emb, image_ids, batch_size=512):
    
    """
    image_emb : Tensor [N, D]
    image_ids : list of image_ids
    """
    image_emb = F.normalize(image_emb, dim=1)

    image_emb = image_emb.cuda()

    # build image_id → list of indices
    id_to_indices = {}
    for idx, img_id in enumerate(image_ids):
        id_to_indices.setdefault(img_id, []).append(idx)

    N = image_emb.size(0)
    ranks = []

    for i in range(0, N, batch_size):
        img_batch = image_emb[i:i+batch_size]    # [B, D]
        sim = img_batch @ image_emb.T             # [B, N]

        sorted_idx = sim.argsort(dim=1, descending=True)

        for j in range(sorted_idx.size(0)):
            query_idx = i + j
            img_id = image_ids[query_idx]

            # same ID images excluding itself
            gt_indices = [idx for idx in id_to_indices[img_id] if idx != query_idx]

            if len(gt_indices) == 0:
                continue  # only one image per ID → skip

            rank = min(
                (sorted_idx[j] == gt).nonzero(as_tuple=True)[0].item()
                for gt in gt_indices
            )
            ranks.append(rank)

    if len(ranks) == 0:
        return {"R@1": 0.0, "R@5": 0.0, "R@10": 0.0}

    ranks = torch.tensor(ranks, device=image_emb.device)

    return {
        "R@1":  (ranks < 1).float().mean().item(),
        "R@5":  (ranks < 5).float().mean().item(),
        "R@10": (ranks < 10).float().mean().item()
    }

In [195]:
@torch.no_grad()
def generate_caption_transformer(image, encoder, decoder, vocab, max_len=30):
    encoder.eval()
    decoder.eval()

    bos = vocab.word2idx["<start>"]
    eos = vocab.word2idx["<end>"]

    image = image.unsqueeze(0).to(device)
    memory = encoder(image)  # [1, N, D]

    generated = torch.tensor([[bos]], device=device)

    for _ in range(max_len - 1):
        logits = decoder(memory, generated)      # [1, t, V]
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # [1,1]
        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == eos:
            break

    words = []
    for idx in generated.squeeze(0).tolist():
        w = vocab.idx2word[idx]
        if w in ("<start>", "<pad>"):
            continue
        if w == "<end>":
            break
        words.append(w)

    return " ".join(words)


In [196]:
image_emb, image_ids  = extract_image_embeddings(test_loader, encoder, decoder, device)


100%|██████████| 6333/6333 [20:58<00:00,  5.03it/s]


In [197]:
text_emb, text_ids = extract_text_embeddings(encoder, decoder, test_loader, vocab, device)

  0%|          | 0/6333 [00:00<?, ?it/s]/home/madhu-thiramdas/ai-work/ai-venv-py310/lib/python3.10/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|██████████| 6333/6333 [21:18<00:00,  4.95it/s]


In [198]:
print("Sample image_ids:", image_ids[:5])
print("Sample text_ids :", text_ids[:5])
print("image_ids unique:", len(set(image_ids)))
print("text_ids unique :", len(set(text_ids)))

Sample image_ids: [391895, 391895, 391895, 391895, 391895]
Sample text_ids : [391895, 391895, 391895, 391895, 391895]
image_ids unique: 40504
text_ids unique : 40504


In [199]:
print("Image → Text:", image_to_text_retrieval(image_emb, text_emb, image_ids, text_ids))
print("Text → Image:", text_to_image_retrieval(image_emb, text_emb, image_ids, text_ids))
print("Text → Text :", text_to_text_retrieval(text_emb, text_ids))
print("Image → Image:", image_to_image_retrieval(image_emb, image_ids))

Image → Text: {'R@1': 0.0, 'R@5': 2.9607113901874982e-05, 'R@10': 5.4279709729598835e-05}
Text → Image: {'R@1': 0.0, 'R@5': 3.4541633795015514e-05, 'R@10': 5.4279709729598835e-05}
Text → Text : {'R@1': 0.00033554728724993765, 'R@5': 0.48983490467071533, 'R@10': 0.5648543834686279}
Image → Image: {'R@1': 0.8001371622085571, 'R@5': 1.0, 'R@10': 1.0}
